# Simple VLA hands-on

標準フローは `Genesis collect -> LeRobot load -> BC/Transformer/Flow train -> Genesis rollout eval -> Qwen connector` です。Genesis/Qwen が使えない環境では同じキー構造の toy fallback を使い、章の接続面だけ先に確認します。


In [ ]:
try:
    import google.colab
    ip = get_ipython()
    ip.run_line_magic("cd", "/content")
    ip.system("rm -rf simple-vla")
    ip.system("git clone https://github.com/Azuma413/simple-vla.git")
    ip.run_line_magic("cd", "/content/simple-vla")
    ip.system("curl -LsSf https://astral.sh/uv/install.sh | sh")
    ip.system("uv pip install --system -e .")
except ImportError:
    print("Not running on Colab; skipping Colab setup.")


In [ ]:
from pathlib import Path

import torch
from torch.utils.data import DataLoader

from env import TinyPickPlaceConfig, GenesisFrankaPickPlaceEnv, FRANKA_QPOS_ACTION_DIM
from part2_vision import GenesisRenderConfig, GenesisRenderDataset, ColoredSquareDataset, SmallVisionEncoder, train_vision_epoch, evaluate_vision
from part1_simulator import (
    TinyPickPlaceDataset, LeRobotPickPlaceDataset,
    collect_genesis_franka_lerobot_dataset, collate_tiny_pick_place, rollout_policy,
)
from part3_bc import StateMLPPolicy, ImageBCPolicy, train_bc_epoch, evaluate_bc_mse, evaluate_bc_rollout
from part4_transformer import ChunkTransformerPolicy, train_chunk_epoch, evaluate_chunk_mse, evaluate_chunk_rollout, export_attention_map, chunk_boundary_jitter
from part5_flow import FlowMatchingDiT, train_flow_epoch, evaluate_flow_rollout, sample_multiple_trajectories, compare_flow_vs_mse_samples
from part6_vla import VLAConnectorPolicy, QwenVLMConfig, connector_optimizer, train_vla_connector_epoch, qwen_forward_smoke

device = "cuda" if torch.cuda.is_available() else "cpu"
device


## 第1部: Genesis render dataset

標準は Genesis render です。Genesis が未導入のときだけ `ColoredSquareDataset` に降格します。照明・背景・カメラ揺らぎは `GenesisRenderConfig` で振ります。


In [ ]:
try:
    vision_train_set = GenesisRenderDataset(GenesisRenderConfig(n=128, image_size=96, backend="gpu" if device == "cuda" else "cpu"))
    vision_test_set = GenesisRenderDataset(GenesisRenderConfig(n=64, image_size=96, seed=1, lighting_range=(0.4, 1.6)))
except Exception as exc:
    print("Genesis render unavailable; using square fallback:", type(exc).__name__, exc)
    vision_train_set = ColoredSquareDataset(n=256, seed=0)
    vision_test_set = ColoredSquareDataset(n=128, seed=1, lighting=0.6)

vision_train = DataLoader(vision_train_set, batch_size=32, shuffle=True)
vision_test = DataLoader(vision_test_set, batch_size=64)
vision = SmallVisionEncoder().to(device)
opt = torch.optim.Adam(vision.parameters(), lr=1e-3)
for epoch in range(2):
    loss = train_vision_epoch(vision, vision_train, opt, device)
    print(epoch, loss, evaluate_vision(vision, vision_test, device))


## 第2部: Genesis collect -> LeRobot load

デモは Franka qpos target を `action` として LeRobot 形式に保存します。`scripted_grasp=False` が標準で、決定的な授業用デモだけ `True` にします。


In [ ]:
data_root = Path("data/genesis_franka_pick_place")
repo_id = "local/simple-vla-genesis-franka"

# 重いので初回だけ実行してください。
# collect_genesis_franka_lerobot_dataset(
#     root=data_root, repo_id=repo_id, n_episodes=8, steps_per_segment=4,
#     backend="gpu" if device == "cuda" else "cpu", image_size=96, scripted_grasp=False,
# )

try:
    train_data = LeRobotPickPlaceDataset(data_root, repo_id=repo_id, chunk_size=4)
    collate_fn = collate_tiny_pick_place
except Exception as exc:
    print("LeRobot dataset unavailable; using tiny fallback:", type(exc).__name__, exc)
    config = TinyPickPlaceConfig(n_episodes=256, seed=2)
    train_data = TinyPickPlaceDataset(config=config, chunk_size=4)
    collate_fn = collate_tiny_pick_place

train_loader = DataLoader(train_data, batch_size=16, shuffle=True, collate_fn=collate_fn)
batch = next(iter(train_loader))
action_dim = batch["action"].shape[-1]
state_dim = batch["state"].shape[-1]
{k: (v.shape if isinstance(v, torch.Tensor) else v[:2]) for k, v in batch.items() if k in {"image", "state", "action", "action_chunk", "instruction"}}


## 第3部: BC train/eval

学習入力は LeRobot adapter 由来の `image/state/action` です。Genesis データなら `action_dim=9` の Franka qpos target、toy fallback なら `action_dim=2` になります。


In [ ]:
state_policy = StateMLPPolicy(state_dim=state_dim, action_dim=action_dim).to(device)
state_opt = torch.optim.Adam(state_policy.parameters(), lr=1e-3)
for epoch in range(2):
    loss = train_bc_epoch(state_policy, train_loader, state_opt, input_key="state", device=device)
    print("state", epoch, loss, evaluate_bc_mse(state_policy, train_loader, input_key="state", device=device))

image_policy = ImageBCPolicy(action_dim=action_dim).to(device)
image_opt = torch.optim.Adam(image_policy.parameters(), lr=1e-3)
for epoch in range(2):
    loss = train_bc_epoch(image_policy, train_loader, image_opt, input_key="image", device=device)
    print("image", epoch, loss, evaluate_bc_mse(image_policy, train_loader, input_key="image", device=device))


## 第4部: Transformer + 可視化

同じ LeRobot batch から `action_chunk` を作り、条件トークン + アクショントークンを単一ストリームで処理します。attention map と chunk jitter を最低限確認します。


In [ ]:
chunk_policy = ChunkTransformerPolicy(action_dim=action_dim, chunk_size=batch["action_chunk"].shape[1]).to(device)
chunk_opt = torch.optim.Adam(chunk_policy.parameters(), lr=1e-3)
for epoch in range(2):
    loss = train_chunk_epoch(chunk_policy, train_loader, chunk_opt, device=device)
    print(epoch, loss, evaluate_chunk_mse(chunk_policy, train_loader, device=device))

with torch.no_grad():
    pred_chunk = chunk_policy(batch["image"][:1].to(device)).cpu()
attention = export_attention_map(chunk_policy.cpu(), batch["image"][0])
chunk_policy.to(device)
attention.shape, chunk_boundary_jitter(pred_chunk)


## 第5部: Flow Matching DiT + 複数サンプル

Flow は同一観測から複数の action chunk をサンプルし、多峰性を可視化・数値化できます。


In [ ]:
flow = FlowMatchingDiT(action_dim=action_dim, chunk_size=batch["action_chunk"].shape[1]).to(device)
flow_opt = torch.optim.Adam(flow.parameters(), lr=1e-3)
for epoch in range(2):
    loss = train_flow_epoch(flow, train_loader, flow_opt, device=device)
    print(epoch, loss)

flow_samples = sample_multiple_trajectories(flow, batch["image"][0], n_samples=4, steps=4)
compare_flow_vs_mse_samples(pred_chunk[0], flow_samples), flow_samples.shape


## Genesis rollout eval

Genesis が使える環境では `rollout_policy(policy, genesis_env, initial_states)` で Franka 環境上の成功率を測ります。toy fallback では同じ呼び出しが tiny world に落ちます。


In [ ]:
try:
    genesis_env = GenesisFrankaPickPlaceEnv(TinyPickPlaceConfig(horizon=24, action_dim=FRANKA_QPOS_ACTION_DIM), image_size=96, backend="gpu" if device == "cuda" else "cpu")
    initial_states = [train_data.episode_initial_state(i) for i in range(min(4, train_data.config.n_episodes))]
    print(evaluate_chunk_rollout(chunk_policy, train_data, n_episodes=len(initial_states), device=device, env=genesis_env, initial_states=initial_states))
except Exception as exc:
    print("Genesis rollout unavailable; using fallback:", type(exc).__name__, exc)
    print(evaluate_chunk_rollout(chunk_policy, train_data, n_episodes=8, device=device))


## 第6部: Qwen connector

VLM と DiT 本体は freeze し、optimizer には connector だけを渡します。実機確認用に 1 枚画像の Qwen forward smoke も用意しています。


In [ ]:
# Qwen smoke はモデル download/VRAM が必要です。
# qwen_forward_smoke(model_id="Qwen/Qwen3.5-0.8B")

# 実学習時:
# vla = VLAConnectorPolicy(qwen_config=QwenVLMConfig(model_id="Qwen/Qwen3.5-0.8B"), action_dim=action_dim, chunk_size=batch["action_chunk"].shape[1]).to(device)
# vla_opt = connector_optimizer(vla, lr=1e-3)
# for epoch in range(2):
#     print(train_vla_connector_epoch(vla, train_loader, vla_opt, device=device))
